# actas-cnn — pipeline didactico end-to-end

Recorre el pipeline completo sobre una acta de muestra para mostrar
cada transformacion intermedia con figuras:

1. PDF original (escaneo ONPE) -> PNG renderizado a 200 dpi.
2. Deteccion de 15 fiducial markers para registracion afin.
3. Recorte por plantilla calibrada en 42 campos (38 partidos + 4 totales).
4. Subdivision de cada campo en 3-4 celdas (una por digito).
5. Filtrado de celdas vacias por convencion right-justified ONPE.
6. Manifest CSV listo para entrenamiento.
7. Entrenamiento de ResNet-18 estilo CIFAR (20 epochs, MPS).
8. Evaluacion downstream: digit / field / acta-level + reconstruccion
   del total agregado.
9. Matriz de confusion + ranking de actas peor reconstruidas.

Tiempo total ~35 min en MacBook Air M2 (MPS).

**Metricas oficiales esperadas (val set, 693 actas, 29,106 campos)**:

| Metrica | Valor |
|---|---|
| Digit-level accuracy | ~98.12% |
| Field-level accuracy | ~98.87% |
| Acta-level accuracy | ~90.33% |
| Reconstruccion exacta del total | ~93.80% |

El notebook asume que `data/` ya esta poblada con crops + manifests +
parquets (ejecutar `bash scripts/run_week1_clean_pipeline.sh` desde la
raiz del repo o bajar el bundle de
[`f3r21/actas-cnn-dataset`](https://huggingface.co/datasets/f3r21/actas-cnn-dataset)).

In [ ]:
# Setup: paths + imports
import os, sys
from pathlib import Path

# Si el notebook se abre desde notebooks/, subir un nivel al repo root
REPO = Path.cwd()
if REPO.name == 'notebooks':
    REPO = REPO.parent
    os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('repo root:', REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import fitz  # PyMuPDF
import torch

from extract_crops import crop_fields, split_digits, es_celda_escrita, load_templates
from scripts.detect_fiducials import detect_15, load_anchors, transform_template
from scripts.build_crops import int_to_digits
try:
    from lakehouse.field_mapping import valor_oficial_para
except ImportError:  # repos sin la capa lakehouse (main antes del merge)
    from scripts.build_crops import field_value_for as valor_oficial_para
from model import build_model
from dataset import default_transforms
from env import torch_device

ARCHIVO_ID = '69e09e9dd7b6147f63eafd1c'  # acta de muestra (val set)
PDF_PATH = REPO / 'sample_data' / 'sample_acta.pdf'
PNG_PATH = REPO / 'sample_data' / 'sample_acta.png'
TEMPLATE = load_templates(REPO / 'templates.json')['presidencial']
ANCHORS = load_anchors(REPO / 'fiducial_anchors.json')

print(f'archivoId: {ARCHIVO_ID}')
print(f'template: {len(TEMPLATE["fields"])} campos')
print(f'anchors: {len(ANCHORS)} roles')

## Paso 1: PDF -> PNG

Cada acta ONPE viene como PDF escaneado. La renderizamos a PNG de 2339 × 3309 px
(equivalente a 200 dpi sobre A4). Si la pagina viene en landscape (auto-detectado
por aspect ratio) se rota 90° para normalizar la orientacion.

In [ ]:
# Renderizar PDF -> PNG (logica equivalente a pdf_to_images.py)
DPI = 200
TARGET_W, TARGET_H = 2339, 3309

doc = fitz.open(PDF_PATH)
page = doc[0]
mat = fitz.Matrix(DPI / 72, DPI / 72)
pix = page.get_pixmap(matrix=mat, alpha=False)
img = Image.frombytes('RGB', (pix.width, pix.height), pix.samples)
if img.width > img.height:  # landscape -> rotar 90°
    img = img.rotate(-90, expand=True)
img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
img.save(PNG_PATH)
doc.close()

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(img, cmap='gray')
ax.set_title(f'sample_acta.pdf -> PNG {img.size}', fontsize=11)
ax.axis('off')
plt.tight_layout(); plt.show()
print(f'PNG guardado en: {PNG_PATH.relative_to(REPO)}')

## Paso 2: Deteccion de 15 fiducial markers

ONPE imprime 15 cuadraditos negros en el perimetro de cada acta:

- 4 esquinas (TL, TR, BL, BR)
- 3 en el margen superior (T1, T2, T3)
- 4 en el margen izquierdo (L1..L4)
- 4 en el margen derecho (R1..R4)

El detector zonal escanea 4 regiones rectangulares de la imagen,
encuentra blobs cuadrados oscuros (filtros por area y aspect ratio), y
los ordena por posicion para asignar el rol. Despues estos markers se
usan para computar una **transformacion afin** que alinea el acta al
espacio canonico de la plantilla.

In [ ]:
markers = detect_15(PNG_PATH)
print(f'detectados: {len(markers)}/15 markers')
for role in ['TL', 'T1', 'T2', 'T3', 'TR', 'L1', 'L2', 'L3', 'L4',
             'R1', 'R2', 'R3', 'R4', 'BL', 'BR']:
    xy = markers.get(role)
    print(f'  {role}: ({xy[0]:4d}, {xy[1]:4d})' if xy else f'  {role}: MISSING')

fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(img, cmap='gray')
for role, (x, y) in markers.items():
    ax.add_patch(plt.Circle((x, y), 35, color='red', fill=False, linewidth=2))
    ax.annotate(role, (x + 40, y), color='red', fontsize=8)
ax.set_title(f'{len(markers)}/15 fiducial markers detectados', fontsize=11)
ax.axis('off')
plt.tight_layout(); plt.show()

## Paso 3: Recorte por plantilla calibrada en 42 campos

La plantilla `templates.json` define las cajas (coordenadas relativas
`[x0, y0, x1, y1]` en `[0, 1]`) de los 42 campos del acta presidencial:

- 38 organizaciones politicas (`partido_01` ... `partido_38`).
- 1 votos en blanco.
- 1 votos nulos.
- 1 votos impugnados.
- 1 total de ciudadanos votantes (4 digitos).

Si la deteccion de fiduciales recupera >=4 markers en >=3 zonas
geometricas, se computa una afin que corrige pequenas diferencias de
escala / rotacion / traslacion entre actas. Si no, se cae al template
original sin alinear.

In [ ]:
tmpl_aligned = transform_template(TEMPLATE, markers, ANCHORS, img.size)
w, h = img.size

fig, ax = plt.subplots(figsize=(7, 10))
ax.imshow(img, cmap='gray')
colors = plt.cm.tab20(np.linspace(0, 1, 42))
for color, f in zip(colors, tmpl_aligned['fields']):
    x0, y0, x1, y1 = f['box']
    rect = mpatches.Rectangle((x0 * w, y0 * h), (x1 - x0) * w, (y1 - y0) * h,
                              linewidth=1.5, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
ax.set_title(f'42 campos del template (post-afin con {len(markers)} markers)', fontsize=11)
ax.axis('off')
plt.tight_layout(); plt.show()

## Paso 4: Subdivision de cada campo en celdas

Cada campo se parte en `n_digits` celdas equiespaciadas (3 para partidos
y votos especiales, 4 para total_ciudadanos). Cada celda es candidata a
contener un digito manuscrito 0-9.

In [ ]:
# Tomar un campo de ejemplo y mostrar su split en celdas
fields_crops = crop_fields(PNG_PATH, tmpl_aligned)
field_name = 'partido_21'  # un partido con cifras visibles tipicamente
field_img = fields_crops[field_name]
n_cells = next(f['n_digits'] for f in TEMPLATE['fields'] if f['name'] == field_name)
cells = split_digits(field_img, n_cells)

fig, axes = plt.subplots(1, 1 + n_cells, figsize=(2.5 * (1 + n_cells), 2.5))
axes[0].imshow(field_img, cmap='gray'); axes[0].set_title(f'{field_name}\n(campo completo)')
axes[0].axis('off')
for i, c in enumerate(cells):
    axes[i + 1].imshow(c, cmap='gray')
    axes[i + 1].set_title(f'celda c{i+1}')
    axes[i + 1].axis('off')
plt.tight_layout(); plt.show()

## Paso 5: Filtrado de celdas vacias por convencion right-justified

ONPE escribe las cifras alineadas a la derecha y deja en blanco las
celdas de la izquierda que correspondan a leading zeros. La funcion
`es_celda_escrita(value, n_cells, cell_position)` retorna `True` si la
celda DEBE contener un digito segun el valor real del parquet.

Ejemplos en un campo de 3 celdas:

- `value=5`     -> `[vacio, vacio, '5']` (solo c3)
- `value=18`    -> `[vacio, '1', '8']`   (c2 y c3)
- `value=144`   -> `['1', '4', '4']`     (todas)
- `value=0`     -> `[vacio, vacio, vacio]` (ninguna; nadie escribe '000')

Esto descarta ~70% de las celdas (las que estan en blanco), evitando
que el dataset este dominado por crops vacios.

In [ ]:
# Cargar parquets para obtener el valor real del campo elegido
archivos = pd.read_parquet(REPO / 'data/labels/actas_archivos.parquet')
votos = pd.read_parquet(REPO / 'data/labels/actas_votos.parquet')
cabecera = pd.read_parquet(REPO / 'data/labels/actas_cabecera.parquet')
id_acta = int(archivos[archivos.archivoId == ARCHIVO_ID].iloc[0]['idActa'])
votos_acta = votos[votos.idActa == id_acta]
total_emit = int(cabecera[cabecera.idActa == id_acta].iloc[0]['totalVotosEmitidos'])
valor_real = valor_oficial_para(field_name, votos_acta, total_emit)
labels = int_to_digits(valor_real, n_cells)
print(f'valor real del campo {field_name}: {valor_real}')
print(f'digitos right-justified en {n_cells} celdas: {labels}')
for pos in range(n_cells):
    escrita = es_celda_escrita(valor_real, n_cells, pos)
    print(f'  c{pos+1}: label={labels[pos]}  es_celda_escrita={escrita}')

fig, axes = plt.subplots(1, n_cells, figsize=(2.5 * n_cells, 2.5))
for pos in range(n_cells):
    escrita = es_celda_escrita(valor_real, n_cells, pos)
    axes[pos].imshow(cells[pos], cmap='gray')
    title = f'c{pos+1}: label={labels[pos]}\n' + ('ESCRITA' if escrita else 'vacia (filtrada)')
    axes[pos].set_title(title, fontsize=9, color='red' if not escrita else 'green')
    axes[pos].axis('off')
plt.tight_layout(); plt.show()

## Paso 6: Dataset listo — manifest CSV

Aplicando los pasos 1-5 sobre 5,000 actas Presidenciales obtenemos
152,000 crops 32×32 etiquetados, distribuidos en 3 splits 70/15/15 por
`archivoId` (no por celda) para evitar fugas. Cada split tiene su
manifest CSV con columnas `path,label`.

In [ ]:
manifest = pd.read_csv(REPO / 'data/manifest_train.csv')
print(f'train: {len(manifest):,} crops')
print('\nprimeras 5 filas:')
print(manifest.head().to_string(index=False))
print('\ndistribucion por clase:')
dist = manifest['label'].value_counts().sort_index()
print(dist.to_string())

fig, ax = plt.subplots(figsize=(8, 3))
dist.plot.bar(ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('clase (digito 0-9)'); ax.set_ylabel('# crops')
ax.set_title(f'Distribucion de clases en train ({len(manifest):,} crops)')
for c, v in dist.items():
    ax.text(c, v + 200, f'{100*v/len(manifest):.1f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.show()

## Paso 7: Entrenamiento — ResNet-18 estilo CIFAR

**Modelo**: ResNet-18 (He et al., 2015) adaptada a entrada 1×32×32:
- Stem `Conv2d(1, 64, 3×3, stride=1)` sin MaxPool inicial (preserva
  resolucion en imagenes chicas).
- 4 etapas residuales (canales 64 -> 128 -> 256 -> 512).
- Global Average Pool al final, Linear(512, 10).
- 11.17M parametros.

**Entrenamiento** (combinacion ganadora de ablations):
- Adam lr=5e-4, batch=128, 20 epochs.
- Label smoothing 0.1 + RandAugment + mixup α=0.2 + cosine LR schedule.
- Device: MPS (M2) / CUDA / CPU autodetectado.

Tiempo esperado: ~17 min en MPS.

In [ ]:
# Entrenamiento full via subprocess para mantener el output visible aqui
!python train.py --manifest data/manifest_train.csv --root data/crops_train \
                 --arch resnet18 --epochs 20 \
                 --label-smoothing 0.1 --randaugment \
                 --mixup 0.2 --cosine-lr \
                 --suffix demo

## Paso 8: Evaluacion downstream

`scripts/evaluate.py` mide lo que realmente importa para el proyecto:

- **Digit-level**: 1 crop predicho correctamente.
- **Field-level**: el entero reconstruido (3-4 digitos) coincide con el
  valor real del parquet.
- **Acta-level**: los 42 campos correctos en simultaneo.
- **Reconstruccion del total agregado**: `sum(partidos) + blanco +
  nulos + impugnados` vs `totalVotosEmitidos` oficial.

El resultado incluye matriz de confusion, histograma de errores y
ranking de las 20 peores actas reconstruidas.

In [ ]:
!python scripts/evaluate.py --split val \
    --checkpoint checkpoints/resnet18_demo_best.pt \
    --out-csv data/evaluate_val_demo.csv

## Paso 9: Visualizaciones de error

Matriz de confusion 10×10 (digit-level) y top-5 actas peor reconstruidas.
Los outputs `data/visualizaciones/evaluate_confusion_val.png` y
`data/visualizaciones/evaluate_error_hist_val.png` se generan
automaticamente; los reusamos aqui.

In [ ]:
# Matriz de confusion + histograma de errores
conf_png = REPO / 'data/visualizaciones/evaluate_confusion_val.png'
hist_png = REPO / 'data/visualizaciones/evaluate_error_hist_val.png'
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].imshow(Image.open(conf_png)); axes[0].axis('off'); axes[0].set_title('Matriz de confusion')
axes[1].imshow(Image.open(hist_png)); axes[1].axis('off'); axes[1].set_title('Distribucion de error del total')
plt.tight_layout(); plt.show()

# Top 5 actas peor reconstruidas
worst = pd.read_csv(REPO / 'data/evaluate_worst20_val.csv').head(5)
print('Top 5 actas peor reconstruidas:')
print(worst[['archivoId', 'total_pred', 'total_real', 'error_total', 'n_fields_wrong']].to_string(index=False))

## Cierre — Resumen y referencias

Reproduccion exitosa del pipeline end-to-end sobre 5,000 actas
presidenciales ONPE (~10% del bucket publico de 871k PDFs).

### Metricas finales (val set, 693 actas, 29,106 campos)

| Metrica | Valor esperado |
|---|---|
| Digit-level accuracy | ~98.12% |
| Field-level accuracy | ~98.87% |
| **Acta-level (42 campos correctos)** | **~90.33%** |
| **Reconstruccion exacta del total agregado** | **~93.80%** |
| MAE del total agregado | ~2.40 votos |

Pequenas diferencias (~±0.5pp) son esperables por la semilla del
`random_split` interno de `train.py`.

### Referencias

- **Datos**: Oficina Nacional de Procesos Electorales (ONPE), Peru.
- **Arquitectura**: He, K., Zhang, X., Ren, S., & Sun, J. (2015).
  *Deep Residual Learning for Image Recognition*. arXiv:1512.03385.
- **Curso**: Topicos en Inteligencia Artificial (CCOMP9-1), 2026.